# FIFA World Cup Standings ETL Pipeline

Pulls group-stage standings from API-Football, transforms them into a DataFrame,
and upserts them into a local MySQL database.

In [1]:
import os          # environment variables
import json         # JSON parsing
import requests     # HTTP requests
import pandas as pd # dataframes
from mysql import connector  # MySQL connection
from dotenv import load_dotenv

In [2]:
load_dotenv()  # load variables from .env

True

In [3]:
API_KEY = os.getenv("API_KEY")
API_HOST = os.getenv("API_HOST")
SEASON = 2022      # free plan supports seasons 2022-2024
LEAGUE_ID = 1       # FIFA World Cup (senior men's)

In [4]:
url = f"https://{API_HOST}/standings"

headers = {
    "x-apisports-key": API_KEY
}

querystring = {"league": LEAGUE_ID, "season": SEASON}

# EXTRACT

In [5]:
response = requests.get(url=url, headers=headers, params=querystring)

In [6]:
payload = response.json()  # parse the JSON response

# TRANSFORM

In [7]:
standings_list = payload['response'][0]['league']['standings']

In [8]:
rows = []
column_names = ['season', 'position', 'team_id', 'team', 'played', 'won', 'draw', 'lost', 'goals_for', 'goals_against', 'goals_diff', 'points', 'form']

for group in standings_list:
    for team in group:
        row = {
            'season':           SEASON,
            'position':         team['rank'],
            'team_id':          team['team']['id'],
            'team':             team['team']['name'],
            'played':           team['all']['played'],
            'won':              team['all']['win'],
            'draw':             team['all']['draw'],
            'lost':             team['all']['lose'],
            'goals_for':        team['all']['goals']['for'],
            'goals_against':    team['all']['goals']['against'],
            'goals_diff':       team['goalsDiff'],
            'points':           team['points'],
            'form':             team['form']
        }
        rows.append(row)

In [9]:
df = pd.DataFrame(rows, columns=column_names)

In [10]:
df.head(20)

,season,position,team_id,team,played,won,draw,lost,goals_for,goals_against,goals_diff,points,form
0,2022,1,1118,Netherlands,3,2,1,0,5,1,4,7,WDW
1,2022,2,13,Senegal,3,2,0,1,5,4,1,6,WWL
2,2022,3,2382,Ecuador,3,1,1,1,4,3,1,4,LDW
3,2022,4,1569,Qatar,3,0,0,3,1,7,-6,0,LLL
4,2022,1,10,England,3,2,1,0,9,2,7,7,WDW
5,2022,2,2384,USA,3,1,2,0,2,1,1,5,WDD
6,2022,3,22,Iran,3,1,0,2,4,7,-3,3,LWL
7,2022,4,767,Wales,3,0,1,2,1,6,-5,1,LLD
8,2022,1,26,Argentina,3,2,0,1,5,2,3,6,WWL
9,2022,2,24,Poland,3,1,1,1,2,2,0,4,LWD


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   season         32 non-null     int64
 1   position       32 non-null     int64
 2   team_id        32 non-null     int64
 3   team           32 non-null     str  
 4   played         32 non-null     int64
 5   won            32 non-null     int64
 6   draw           32 non-null     int64
 7   lost           32 non-null     int64
 8   goals_for      32 non-null     int64
 9   goals_against  32 non-null     int64
 10  goals_diff     32 non-null     int64
 11  points         32 non-null     int64
 12  form           32 non-null     str  
dtypes: int64(11), str(2)
memory usage: 3.4 KB


# LOAD

In [12]:
MYSQL_HOST = os.getenv("MYSQL_HOST")
MYSQL_PORT = os.getenv("MYSQL_PORT")
MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE")

In [13]:
db_connection = connector.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=MYSQL_DATABASE,
)

cursor = db_connection.cursor()
print("Connected successfully!") if db_connection.is_connected() else print("Connection failed")

Connected successfully!


In [14]:
sql_table = "standings"
cursor.execute("SHOW TABLES LIKE %s", (f"{sql_table}",))

if cursor.fetchone() is None:
    print(f"Table '{sql_table}' does not exist.")
else:
    print(f"Table '{sql_table}' exists.")

Table 'standings' exists.


In [15]:
table_cols = ['season', 'position', 'team_id', 'team', 'played', 'won', 'draw', 'lost', 'goals_for', 'goals_against', 'goals_diff', 'points', 'form']

standings_df = df[table_cols]

In [16]:
standings_records_tuples = standings_df.itertuples(index=False, name=None)  # convert to tuples for insertion
list_of_standings_records = list(standings_records_tuples)

In [17]:
UPSERT_QUERY = f"""
INSERT INTO {sql_table} (season, position, team_id, team, played, won, draw, lost, goals_for, goals_against, goals_diff, points, form)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s) as src
ON DUPLICATE KEY UPDATE
    position                = src.position,
    team_id                 = src.team_id,
    team                    = src.team,
    played                  = src.played,
    won                     = src.won,
    draw                    = src.draw,
    lost                    = src.lost,
    goals_for               = src.goals_for,
    goals_against           = src.goals_against,
    goals_diff              = src.goals_diff,
    points                  = src.points,
    form                    = src.form
"""

In [18]:
no_of_rows_uploaded_to_mysql = len(list_of_standings_records)

In [19]:
try:
    cursor.executemany(UPSERT_QUERY, list_of_standings_records)
    db_connection.commit()  # save changes
    print(f"Successfully uploaded {no_of_rows_uploaded_to_mysql} rows to the '{sql_table}' table.")
except Exception as e:
    print(f"Error occurred while uploading data: {e}")
finally:
    cursor.close()
    db_connection.close()
    print("Database connection closed.")

Successfully uploaded 32 rows to the 'standings' table.
Database connection closed.


# RESULTS

In [20]:
# Reconnect and pull the data back to confirm the upload landed correctly
check_connection = connector.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=MYSQL_DATABASE,
)

results_df = pd.read_sql(f"SELECT * FROM {sql_table} ORDER BY season, position", check_connection)

check_connection.close()
results_df

C:\Users\schem\AppData\Local\Temp\ipykernel_107496\2349300296.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  results_df = pd.read_sql(f"SELECT * FROM {sql_table} ORDER BY season, position", check_connection)


,season,position,team_id,team,played,won,draw,lost,goals_for,goals_against,goals_diff,points,form
0,2022,1,2,France,3,2,0,1,6,3,3,6,LWW
1,2022,1,6,Brazil,3,2,0,1,3,1,2,6,LWW
2,2022,1,10,England,3,2,1,0,9,2,7,7,WDW
3,2022,1,12,Japan,3,2,0,1,4,3,1,6,WLW
4,2022,1,26,Argentina,3,2,0,1,5,2,3,6,WWL
5,2022,1,27,Portugal,3,2,0,1,6,4,2,6,LWW
6,2022,1,31,Morocco,3,2,1,0,4,1,3,7,WWD
7,2022,1,1118,Netherlands,3,2,1,0,5,1,4,7,WDW
8,2022,2,3,Croatia,3,1,2,0,4,1,3,5,DWD
9,2022,2,9,Spain,3,1,1,1,9,3,6,4,LDW
